# 03 – Forecasting Models
Ziel: Alle 4 Modelle trainieren und Forecast-Metriken vergleichen.

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..')

from src.forecasting.train import prepare_data, scale_features
from src.forecasting.evaluate_forecast import evaluate_all
from src.models import linear_regression, random_forest, neural_net, quantile_regression
from src.utils.plotting import plot_forecast_vs_actual, plot_metric_comparison

## Daten laden

In [ ]:
df = pd.read_csv('../data/processed/final_dataset.csv', parse_dates=['timestamp'])
print(df.shape)
df.head()

## Train/Test Split
`shuffle=False` ist bei Zeitreihen Pflicht – keine zufällige Aufteilung!

In [ ]:
X_train, X_test, y_train, y_test = prepare_data(df)
X_sc_train, X_sc_test, scaler = scale_features(X_train, X_test)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')
print(f'Features: {X_train.shape[1]}')

## 1. Linear Regression

In [ ]:
lr_model  = linear_regression.train(X_train, y_train, polynomial_degree=2)
y_pred_lr = linear_regression.predict(lr_model, X_test)
plot_forecast_vs_actual(y_test, y_pred_lr, 'Linear Regression')

## 2. Random Forest

In [ ]:
rf_model  = random_forest.train(X_train, y_train)
y_pred_rf = random_forest.predict(rf_model, X_test)
plot_forecast_vs_actual(y_test, y_pred_rf, 'Random Forest')

## 3. Neural Network

In [ ]:
nn_model  = neural_net.train(X_sc_train, y_train, epochs=100)
y_pred_nn = neural_net.predict(nn_model, X_sc_test)
plot_forecast_vs_actual(y_test, y_pred_nn, 'Neural Network')

## 4. Quantile Regression

In [ ]:
qr_models = quantile_regression.train_all_quantiles(X_train, y_train)
y_pred_q50 = quantile_regression.predict_quantile(qr_models, X_test, 0.5)

from src.utils.plotting import plot_quantile_forecast
q10 = quantile_regression.predict_quantile(qr_models, X_test, 0.1)
q90 = quantile_regression.predict_quantile(qr_models, X_test, 0.9)
plot_quantile_forecast(y_test, q10, y_pred_q50, q90)

## Vergleich: Forecast-Metriken
**Wichtig**: Das ist NICHT die finale Bewertung – die kommt in Notebook 04.

In [ ]:
models = {
    'LinearRegression':     lr_model,
    'RandomForest':          rf_model,
    'NeuralNet':             nn_model,
    'QuantileRegression':    qr_models,
}
df_results = evaluate_all(models, X_test, X_sc_test, y_test)
df_results

In [ ]:
plot_metric_comparison(df_results.to_dict('index'), metric='rmse')

## Key Insight
Das RMSE-Ranking sagt noch nichts über den wirtschaftlichen Wert aus.
→ Weiter zu Notebook 04 für die echte Evaluation.